# Exploración de partidos de Champions

Este notebook explora la copia incluida de OpenFootball y muestra cómo construir variables anteriores al partido. El proyecto ya incluye un modelo **Random Forest**; su entrenamiento y evaluación se realizan mediante `scripts/entrenar_modelo.py`, no en este notebook. Consulta el método completo en `docs/MODELO.md`.

Fuente: https://github.com/openfootball/champions-league. Licencia CC0-1.0. Versión exacta y hashes en `data/fuentes.json`.

**Ejecutar nuevamente:** se limpiaron las salidas de la versión anterior para no presentar resultados de un dataset desactualizado. Ejecuta todas las celdas en orden con el CSV actual antes de usar sus resultados como evidencia. Esta actualización del documento no ejecutó el notebook.

En Jupyter, abre el notebook dentro del proyecto. En Google Colab, sube este notebook y, cuando se solicite, el archivo `data/processed/partidos.csv` del proyecto.


In [ ]:
from pathlib import Path
import pandas as pd

candidatos = [Path("../data/processed/partidos.csv"), Path("data/processed/partidos.csv"), Path("partidos.csv")]
ruta = next((p for p in candidatos if p.exists()), None)
if ruta is None:
    from google.colab import files
    print("Sube el archivo partidos.csv incluido en el proyecto.")
    archivos = files.upload()
    ruta = next((Path(nombre) for nombre in archivos if nombre.endswith(".csv")), None)
    if ruta is None:
        raise ValueError("Debes subir partidos.csv")
df = pd.read_csv(ruta, encoding="utf-8-sig", parse_dates=["fecha"])
print("Partidos:", len(df))
print("Equipos:", len(set(df.local) | set(df.visitante)))
print("Fechas:", df.fecha.min().date(), "a", df.fecha.max().date())


## 1. Estructura y controles iniciales
Cada fila representa un partido. Los goles y la etiqueta corresponden a los 90 minutos más tiempo añadido.


In [ ]:
print(df[["fecha", "local", "visitante", "goles_local_90", "goles_visitante_90", "resultado"]].head().to_string(index=False))
print("\nFaltantes por columna:\n", df.isna().sum().to_string())
print("\nIDs duplicados:", df.id.duplicated().sum())


## 2. Cobertura y distribución
La copia actual contiene **1.372 partidos, 96 equipos y 10 temporadas**, de 2016-17 a 2025-26, con resultados hasta el 30/05/2026. La temporada 2019-20 contiene 119 partidos; 2024-25 y 2025-26 contienen 189 cada una; las otras siete contienen 125 cada una.

Los conteos se verifican contra el encabezado de cada archivo original. Que coincidan significa cobertura completa según esa fuente, no una auditoría independiente de todos los resultados. No hay actualización en vivo ni datos de 2026-27 en esta copia.


In [ ]:
print(df.groupby("temporada").size().rename("partidos").to_string())
resumen = df.resultado.value_counts().reindex(["Local", "Empate", "Visitante"]).rename("partidos").to_frame()
resumen["porcentaje"] = (100 * resumen.partidos / len(df)).round(2)
print("\n", resumen.to_string())
print("\nPromedio de goles a 90 min:", round((df.goles_local_90 + df.goles_visitante_90).mean(), 2))


## 3. Tratamiento de prórrogas
El marcador original y el resultado de los 90 minutos se conservan por separado. Una tanda de penales no determina la clase de resultado a 90 minutos.


In [ ]:
print(df.loc[df.hubo_prorroga, ["fecha", "local", "visitante", "marcador_original", "goles_local_90", "goles_visitante_90", "resultado"]].to_string(index=False))


## 4. Ejemplo de variables anteriores al partido
Se filtra por fecha estrictamente anterior al corte antes de seleccionar los últimos cinco partidos. El historial se limita a Champions. No se interpreta como una predicción.


In [ ]:
equipo = "Real Madrid"
corte = pd.Timestamp("2024-04-09")
historial = df[(df.fecha < corte) & ((df.local == equipo) | (df.visitante == equipo))].sort_values("fecha").tail(5).copy()
historial["goles_favor"] = historial.apply(lambda r: r.goles_local_90 if r.local == equipo else r.goles_visitante_90, axis=1)
historial["goles_contra"] = historial.apply(lambda r: r.goles_visitante_90 if r.local == equipo else r.goles_local_90, axis=1)
print(historial[["fecha", "local", "visitante", "goles_favor", "goles_contra"]].to_string(index=False))
print("\nPartidos usados:", len(historial))
print("Promedio de goles a favor:", historial.goles_favor.mean())
print("Promedio de goles en contra:", historial.goles_contra.mean())
assert (historial.fecha < corte).all()


## 5. Modelo implementado y siguiente etapa

El proyecto ya construye seis variables con los últimos cinco partidos de Champions anteriores a cada encuentro: puntos, goles a favor y goles en contra por equipo. No usa el resultado del partido objetivo como entrada. El ejemplo de la sección anterior mantiene el corte histórico 09/04/2024 para mostrar este filtrado; no constituye una predicción.

- **Preparación:** se requieren cinco partidos previos por equipo. Se excluyen como objetivos las finales y los casos de sede no admitidos descritos en `docs/MODELO.md`.
- **Entrenamiento inicial:** temporadas 2016-17 a 2022-23, con 481 partidos utilizables tras las exclusiones.
- **Validación:** temporada 2023-24, con 96 partidos utilizables. Se seleccionó Random Forest por su F1 macro frente a regresión logística; DummyClassifier se conserva como referencia.
- **Modelo evaluado:** ajustado con entrenamiento más validación, 577 partidos, hasta el 08/05/2024.
- **Prueba reservada:** temporada 2024-25, con 154 partidos utilizables. Random Forest obtuvo accuracy de 55,19 % (85 aciertos), frente a 51,30 % (79 aciertos) de la referencia. Su F1 macro fue 0,3808 y su log loss 0,9753.
- **Limitación observada:** no acertó ninguno de los 23 empates reales de prueba. La mejora global no demuestra estabilidad futura.
- **Modelo actual:** reajustado con 895 partidos utilizables, hasta el 06/05/2026. No tiene una prueba futura independiente y no se le atribuyen las métricas del modelo evaluado.

La API propia sirve las estimaciones y la interfaz permite consultar los datos, comparar los últimos partidos, revisar enfrentamientos directos y solicitar una predicción experimental. Los enfrentamientos directos son descriptivos: todavía no son una variable del modelo. Las métricas y sus particiones están en `models/metricas.json`.

El laboratorio ya evaluó 13 variables, incluidas las de empates, fuerza Elo y enfrentamientos directos. En 164 partidos de 2025-26, la regresión logística equilibrada acertó 6 de 27 empates y obtuvo peor log loss que el Random Forest de seis variables. Los resultados están en `docs/MEJORAS_MODELO.md`. Para evaluar el uso futuro, se propone guardar los pronósticos antes de cada partido. El estudiante excluyó las sesiones con compañeros del alcance de esta entrega.

**Limitaciones:** fuente comunitaria sin auditoría independiente de cada resultado, datos hasta el 30/05/2026, ausencia de lesiones y alineaciones y falta de un campo uniforme de sede neutral. Las probabilidades no están calibradas. El objetivo es el resultado a 90 minutos más añadido, no el marcador exacto ni la clasificación de una eliminatoria.
